In [ ]:
# Skin Cancer Classification Project with GPU/MPS Optimization
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, TensorBoard, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Configure hardware acceleration
def configure_hardware():
    gpus = tf.config.list_physical_devices('GPU')
    mps_device = None

    try:
        mps_device = tf.config.list_physical_devices('MPS')
    except:
        pass

    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            tf.config.optimizer.set_jit(True)
            print("Using GPU with XLA acceleration")
        except RuntimeError as e:
            print(e)
    elif mps_device:
        print("Using Apple MPS acceleration")
    else:
        print("Using CPU")

configure_hardware()

# Enable mixed precision training
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

# Project Parameters
IMG_SIZE = 300
BATCH_SIZE = 64
EPOCHS = 50
FINE_TUNE_EPOCHS = 30
NUM_CLASSES = 7
TRAIN_DIR = "./train_structured"
VAL_DIR = "./val_structured"

# Validate data structure
required_subdirs = [str(i) for i in range(7)]
for dir_path in [TRAIN_DIR, VAL_DIR]:
    actual_subdirs = sorted(os.listdir(dir_path))
    assert actual_subdirs == required_subdirs, f"Invalid structure in {dir_path}"

# Data Generators
def create_datagen(augment=False):
    if augment:
        return ImageDataGenerator(
            preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
            rotation_range=45,
            width_shift_range=0.25,
            height_shift_range=0.25,
            shear_range=0.2,
            zoom_range=0.3,
            horizontal_flip=True,
            vertical_flip=True,
            fill_mode='constant',
            brightness_range=[0.8, 1.2]
        )
    return ImageDataGenerator(preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input)

train_datagen = create_datagen(augment=True)
val_datagen = create_datagen()

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale'
)

validation_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale'
)

# Class weights
class_labels = train_generator.classes
present_classes = np.unique(class_labels)
class_weights = np.ones(NUM_CLASSES)
if len(present_classes) > 0:
    temp_weights = compute_class_weight('balanced', classes=present_classes, y=class_labels)
    class_weights[present_classes] = temp_weights
class_weights_dict = {i: class_weights[i] for i in range(NUM_CLASSES)}

# Model architecture
def create_model(alpha=1.0):
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 1))
    x = Lambda(lambda img: tf.repeat(img, 3, axis=-1), output_shape=(IMG_SIZE, IMG_SIZE, 3))(inputs)
    base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet', alpha=alpha)
    base_model.trainable = False
    x = base_model(x)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(1024, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    x = Dropout(0.7)(x)
    x = Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l1_l2(0.005))(x)
    x = Dropout(0.5)(x)
    outputs = Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    return Model(inputs, outputs), base_model

model1, base1 = create_model(alpha=1.0)
model2, base2 = create_model(alpha=0.75)

# Compile and train
def compile_and_train(model, base_model, model_name):
    base_model.trainable = False
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=3e-4,
        decay_steps=len(train_generator)*10,
        decay_rate=0.9
    )
    model.compile(optimizer=Adam(lr_schedule), loss='categorical_crossentropy', metrics=['accuracy'])

    callbacks = [
        EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
        ModelCheckpoint(f"best_{model_name}.h5", save_best_only=True),
        TensorBoard(log_dir=f'logs/{model_name}', profile_batch=0)
    ]

    print(f"\nTraining {model_name} - Phase 1")
    history = model.fit(train_generator, validation_data=validation_generator, epochs=EPOCHS, callbacks=callbacks, class_weight=class_weights_dict)

    print(f"\nFine-tuning {model_name} - Phase 2")
    base_model.trainable = True
    for layer in base_model.layers[:150]:
        layer.trainable = False

    model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
    history_fine = model.fit(train_generator, validation_data=validation_generator, initial_epoch=history.epoch[-1], epochs=EPOCHS + FINE_TUNE_EPOCHS, callbacks=callbacks)

    return history, history_fine

print("Starting Model 1 Training")
hist1, hist_fine1 = compile_and_train(model1, base1, "model1")

print("\nStarting Model 2 Training")
hist2, hist_fine2 = compile_and_train(model2, base2, "model2")

# Plot training results
def plot_training(history, model_name):
    acc = history[0].history['accuracy'] + history[1].history['accuracy']
    val_acc = history[0].history['val_accuracy'] + history[1].history['val_accuracy']
    loss = history[0].history['loss'] + history[1].history['loss']
    val_loss = history[0].history['val_loss'] + history[1].history['val_loss']

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Train Accuracy')
    plt.plot(val_acc, label='Val Accuracy')
    plt.axhline(0.9, linestyle='--', color='r', label='Target 90%')
    plt.title(f"{model_name} Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Train Loss')
    plt.plot(val_loss, label='Val Loss')
    plt.title(f"{model_name} Loss")
    plt.legend()
    plt.show()

plot_training((hist1, hist_fine1), "Model 1")
plot_training((hist2, hist_fine2), "Model 2")

# Final test & evaluation
def final_test(test_dir):
    models = [tf.keras.models.load_model(f"best_model{i+1}.h5") for i in range(2)]
    test_datagen = ImageDataGenerator(preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input)
    test_gen = test_datagen.flow_from_directory(
        test_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        color_mode='grayscale',
        shuffle=False
    )

    preds = [model.predict(test_gen) for model in models]
    final_pred = np.argmax(0.4 * preds[0] + 0.6 * preds[1], axis=1)
    true_labels = test_gen.classes
    class_names = list(train_generator.class_indices.keys())

    print("\nClassification Report:")
    print(classification_report(true_labels, final_pred, target_names=class_names))

    cm = confusion_matrix(true_labels, final_pred)
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix")
    plt.ylabel('True Labels')
    plt.xlabel('Predicted Labels')
    plt.show()

    print("\nClass-wise Accuracy:")
    for i, class_name in enumerate(class_names):
        class_indices = np.where(true_labels == i)[0]
        class_accuracy = accuracy_score(true_labels[class_indices], final_pred[class_indices])
        print(f"{class_name}: {class_accuracy * 100:.2f}%")

    return [class_names[i] for i in final_pred]

# Usage:
# test_results = final_test("./test_structured")

FileNotFoundError: [Errno 2] No such file or directory: './train_structured'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import shutil

def structure_images(source_dir, target_dir, num_classes=7):
    os.makedirs(target_dir, exist_ok=True)

    for i in range(num_classes):
        class_folder = os.path.join(target_dir, str(i))
        os.makedirs(class_folder, exist_ok=True)

    count = 0
    for filename in os.listdir(source_dir):
        if filename.endswith(".jpg"):
            try:
                class_label = filename.split("_")[0]
                if class_label.isdigit() and int(class_label) < num_classes:
                    source_path = os.path.join(source_dir, filename)
                    dest_path = os.path.join(target_dir, class_label, filename)
                    shutil.copy(source_path, dest_path)
                    count += 1
            except Exception as e:
                print(f"⚠ Failed for {filename}: {e}")

    print(f"✅ Moved {count} images into structured directory: {target_dir}")

structure_images('/content/drive/MyDrive/skin classification/train', './train_structured', num_classes=7)
structure_images('/content/drive/MyDrive/skin classification/val', './val_structured', num_classes=7)

✅ Moved 2965 images into structured directory: ./train_structured
✅ Moved 980 images into structured directory: ./val_structured
